# Day 11 — Chi-Square Test (Categorical Variables)
**90-Day Data Science Roadmap | FC Lahore Lions Dataset**

---
### What We're Solving
Are certain positions more likely to get injured? Both columns are **categories** — no means, no t-test. We use **Chi-Square.**

## Step 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## Step 2 — Build FC Lahore Lions Dataset

In [ ]:
np.random.seed(42)          # same seed = same results every run
n = 120                     # 120 players in the squad

In [ ]:
# Randomly assign positions with realistic squad proportions
positions = np.random.choice(
    ['GK', 'DEF', 'MID', 'FWD'],
    size=n,
    p=[0.1, 0.3, 0.35, 0.25]    # 10% GK, 30% DEF, 35% MID, 25% FWD
)
print(positions[:10])            # preview first 10

In [ ]:
# Each position has a different injury probability
injury_prob = {
    'GK':  0.15,    # GK — safest position
    'DEF': 0.45,    # DEF — tackles, physical duels
    'MID': 0.30,    # MID — moderate risk
    'FWD': 0.50     # FWD — most risk, pressing & collisions
}

In [ ]:
# Assign injury outcome per player using their position's probability
injured = []

for p in positions:
    prob_yes = injury_prob[p]                                      # their injury chance
    prob_no  = 1 - prob_yes                                        # chance of staying fit
    result   = np.random.choice(['Yes', 'No'], p=[prob_yes, prob_no])  # random pick
    injured.append(result)

injured = np.array(injured)    # convert list to numpy array

In [ ]:
# Combine into a DataFrame
df = pd.DataFrame({'Position': positions, 'Injured': injured})
print(f"Shape: {df.shape}")
df.head(10)

## Step 3 — Build the Contingency Table (Observed)

In [ ]:
# crosstab = count how many players fall into each Position x Injured box
ct = pd.crosstab(df['Position'], df['Injured'])
print("Observed Table:")
ct

## Step 4 — Run the Chi-Square Test

In [ ]:
# chi2_contingency always returns 4 things — never unpack only 2!
chi2, p, dof, expected = stats.chi2_contingency(ct)

print(f"Chi2 Statistic : {chi2:.4f}")
print(f"p-value        : {p:.4f}")
print(f"Degrees of Freedom: {dof}")

## Step 5 — Print the Expected Table

In [ ]:
# expected comes back as plain array — add labels from ct so it's readable
expected_df = pd.DataFrame(
    expected,
    index=ct.index,        # same row labels as observed table
    columns=ct.columns     # same column labels as observed table
).round(2)

print("Expected Table (if no relationship):")
expected_df

## Step 6 — Observed vs Expected Side by Side

In [ ]:
print("OBSERVED:")
print(ct)
print()
print("EXPECTED (if independent):")
print(expected_df)
print()
print("GAP (Observed - Expected):")
print((ct - expected_df).round(2))

## Step 7 — Conclusion

In [ ]:
alpha = 0.05

print(f"p-value = {p:.4f}")
print(f"alpha   = {alpha}")
print()

if p < alpha:
    print("✅ p < alpha — Reject H0")
    print("Strong evidence that Position and Injury are ASSOCIATED.")
    print("(Not proven. Not causal. Just associated.)")
else:
    print("❌ p >= alpha — Fail to reject H0")
    print("No significant evidence of association.")

## Step 8 — Safety Check (Expected Counts < 5)

In [ ]:
# Chi-square is unreliable if any expected cell count is below 5
if (expected < 5).any():
    print("⚠️ Warning: Some expected counts < 5 — consider Fisher's Exact Test")
else:
    print("✅ All expected counts >= 5 — Chi-Square result is reliable")

---
## Key Takeaways
- **Chi-square** = test for two categorical variables (independence vs association)
- `stats.chi2_contingency()` always returns **4 values** — chi2, p, dof, expected
- p < 0.05 → reject H0 → **strong evidence** of association (never 'proof', never 'cause')
- For **strength** of association → use Cramér's V
- For **cause** → need experiments, not chi-square